In [1]:
pip install bert-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 1: Configuration
import os
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import Dataset
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
from bert_score import score
import nltk
from tqdm import tqdm
# nltk.download('wordnet')

2026-01-18 11:41:53.964770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768736514.147620      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768736514.199773      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768736514.636846      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768736514.636886      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768736514.636888      55 computation_placer.cc:177] computation placer alr

In [5]:
# Configuration dictionary for easy customization
CONFIG = {
    'model_name': 't5-large',
    'max_input_length': 512,
    'max_target_length': 128,
    'batch_size': 1,
    'epochs': 10,
    'learning_rate': 2e-5,
    'test_size': 0.2,
    'random_state': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'output_dir': './t5_model',
    'model_save_path': './t5_model/t5_requirements_to_userstory'
}

In [4]:
def load_and_prepare_data(file_path):
    # Load dataset as pandas first
    df = pd.read_csv(file_path)

    # Clean columns
    df['Functional Requirement'] = df['Functional Requirement'].fillna('').astype(str)
    df['User Story'] = df['User Story'].fillna('').astype(str)

    # Group FRs by User Story ID
    grouped = df.groupby('User Story ID').agg({
        'User Story': 'first',
        'Functional Requirement': lambda x: ' '.join(x)
    }).reset_index()

    # Create input-output pairs
    data = {
        'input_text': ['functional requirement: ' + req for req in grouped['Functional Requirement']],
        'target_text': grouped['User Story']
    }

    # ✅ CREATE HF DATASET FIRST
    full_dataset = Dataset.from_dict(data)

    # ✅ USE HUGGINGFACE SPLIT (this fixes your error)
    split = full_dataset.train_test_split(
        test_size=CONFIG['test_size'],
        seed=CONFIG['random_state']
    )

    train_dataset = split['train']
    test_dataset = split['test']

    return train_dataset, test_dataset


In [ ]:
# Tokenize dataset for T5 model
def tokenize_dataset(dataset, tokenizer):
    def tokenize_function(examples):
        # Tokenize inputs and targets
        inputs = tokenizer(
            examples['input_text'],
            max_length=CONFIG['max_input_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        targets = tokenizer(
            examples['target_text'],
            max_length=CONFIG['max_target_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs.input_ids,  # Keep as tensor, remove squeeze
            'attention_mask': inputs.attention_mask,
            'labels': targets.input_ids
        }
    
    # Apply tokenization and set format for PyTorch tensors
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized_dataset

In [ ]:
# Evaluate model with BLEU, METEOR, and BERTScore
def evaluate_model(model, test_dataset, tokenizer):
    from tqdm import tqdm
    model.eval()
    predictions = []
    references = []
    input_texts = []
    
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset, batch_size=CONFIG['batch_size']
    )
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=CONFIG['max_target_length'] * 3,  # Further increase for complete outputs
                min_length=30,  # Encourage longer, meaningful outputs
                num_beams=8,  # Increase beams for better quality
                length_penalty=0.6,  # Further reduce penalty for longer sequences
                no_repeat_ngram_size=3,  # Prevent repetitive phrases
                early_stopping=True,
                do_sample=True,  # Add sampling for diversity
                top_p=0.9,  # Use nucleus sampling to improve output variety
                temperature=0.7  # Control randomness
            )
            
            preds = [tokenizer.decode(ids, skip_special_tokens=True) for ids in outputs]
            refs = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['labels']]
            inputs = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['input_ids']]
            
            predictions.extend(preds)
            references.extend(refs)
            input_texts.extend(inputs)
    
    # Debug: Print inputs, predictions, and references
    print("\nDebug: Sample Inputs, Predictions, and References:")
    for i, (inp, pred, ref) in enumerate(zip(input_texts[:5], predictions[:5], references[:5])):
        print(f"Sample {i + 1}:")
        print(f"Input (Functional Requirement): {inp}")
        print(f"Predicted User Story: {pred}")
        print(f"Reference User Story: {ref}")
        print()
    
    # Debug: Print counts of valid and empty predictions
    valid_predictions = [p for p in predictions if p.strip()]
    print(f"Total Predictions: {len(predictions)}")
    print(f"Valid (Non-Empty) Predictions: {len(valid_predictions)}")
    print(f"Empty Predictions: {len(predictions) - len(valid_predictions)}")
    
    # Filter valid prediction-reference pairs
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p.strip() and r.strip()]
    if not valid_pairs:
        print("Warning: No valid prediction-reference pairs for metric calculation.")
        return {
            'bleu': 0.0,
            'meteor': 0.0,
            'bertscore_precision': 0.0,
            'bertscore_recall': 0.0,
            'bertscore_f1': 0.0,
            'predictions': predictions,
            'references': references
        }
    
    valid_predictions, valid_references = zip(*valid_pairs)
    
    # Calculate BLEU and METEOR with smoothing
    bleu_scores = []
    meteor_scores = []
    
    for pred, ref in valid_pairs:
        try:
            bleu_scores.append(sentence_bleu([ref.split()], pred.split(), 
                                          weights=(0.25, 0.25, 0.25, 0.25),
                                          smoothing_function=lambda precisions, **kw: [p + 1e-12 for p in precisions]))
            meteor_scores.append(meteor_score([ref.split()], pred.split()))
        except Exception as e:
            print(f"Error calculating metrics for pred='{pred}', ref='{ref}': {e}")
            continue
    
    # Calculate BERTScore
    bertscore_precision, bertscore_recall, bertscore_f1 = 0.0, 0.0, 0.0
    if valid_predictions and valid_references:
        try:
            P, R, F1 = score(list(valid_predictions), list(valid_references), lang='en', verbose=True)
            bertscore_precision = P.mean().item() if P.numel() > 0 else 0.0
            bertscore_recall = R.mean().item() if R.numel() > 0 else 0.0
            bertscore_f1 = F1.mean().item() if F1.numel() > 0 else 0.0
        except Exception as e:
            print(f"Error calculating BERTScore: {e}")
    
    return {
        'bleu': sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0,
        'meteor': sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0,
        'bertscore_precision': bertscore_precision,
        'bertscore_recall': bertscore_recall,
        'bertscore_f1': bertscore_f1,
        'predictions': predictions,
        'references': references
    }

In [ ]:
# Train T5 model with tqdm, table display, multiple GPUs, and early stopping
def train_model(train_dataset, test_dataset, tokenizer):
    import pandas as pd
    from tqdm import tqdm
    import torch
    from torch.nn.parallel import DataParallel
    
    model = T5ForConditionalGeneration.from_pretrained(CONFIG['model_name']).to(CONFIG['device'])
    
    # Wrap model with DataParallel for multiple GPUs if available
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = DataParallel(model)
    
    model = model.to(CONFIG['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
    
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'], shuffle=True
    )
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset, batch_size=CONFIG['batch_size']
    )
    
    # Early stopping parameters
    patience = 3  # Number of epochs to wait for improvement
    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stop = False
    
    # Lists to store losses for table
    epoch_data = []
    
    for epoch in range(CONFIG['epochs']):
        if early_stop:
            print(f"Early stopping triggered after {epoch} epochs.")
            break
            
        model.train()
        total_train_loss = 0
        train_loop = tqdm(train_dataloader, desc=f"Training Epoch {epoch + 1}")
        
        for batch in train_loop:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            # Handle DataParallel loss (mean across GPUs)
            loss = outputs.loss
            if isinstance(model, DataParallel):
                loss = loss.mean()  # Aggregate loss across GPUs
            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()
            
            train_loop.set_postfix({'batch_loss': loss.item()})
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        
        # Compute validation loss
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            val_loop = tqdm(test_dataloader, desc=f"Validation Epoch {epoch + 1}")
            for batch in val_loop:
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                # Handle DataParallel loss for validation
                loss = outputs.loss
                if isinstance(model, DataParallel):
                    loss = loss.mean()  # Aggregate loss across GPUs
                total_val_loss += loss.item()
                val_loop.set_postfix({'batch_loss': loss.item()})
        
        avg_val_loss = total_val_loss / len(test_dataloader)
        
        # Store epoch results
        epoch_data.append({
            'Epoch': epoch + 1,
            'Training Loss': avg_train_loss,
            'Validation Loss': avg_val_loss
        })
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            # Save best model
            os.makedirs(CONFIG['output_dir'], exist_ok=True)
            if isinstance(model, DataParallel):
                model.module.save_pretrained(CONFIG['model_save_path'])
            else:
                model.save_pretrained(CONFIG['model_save_path'])
            tokenizer.save_pretrained(CONFIG['model_save_path'])
            print(f"New best model saved at epoch {epoch + 1} with validation loss: {avg_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                early_stop = True
        
        print(f"Epoch {epoch + 1}: Training Loss = {avg_train_loss:.4f}, Validation Loss = {avg_val_loss:.4f}")
    
    # Display results in a table
    results_df = pd.DataFrame(epoch_data)
    print("\nTraining Progress:")
    print(results_df.to_string(index=False))
    
    # Return the unwrapped model for evaluation
    if isinstance(model, DataParallel):
        return model.module
    return model

In [ ]:
# Main execution flow
# Initialize tokenizer
tokenizer = T5Tokenizer.from_pretrained(CONFIG['model_name'])

# Load and prepare data
dataset_path = "/kaggle/input/merget-userstory/merged_all.csv"  # Path from your input
train_dataset, test_dataset = load_and_prepare_data(dataset_path)

# Debug: Check dataset size and sample
# print(f"Train dataset size: {len(train_dataset)}")
# print(f"Test dataset size: {len(test_dataset)}")
# print("Train dataset sample:", train_dataset[0])
# print("Test dataset sample:", test_dataset[0])

# Tokenize datasets
train_dataset = tokenize_dataset(train_dataset, tokenizer)
test_dataset = tokenize_dataset(test_dataset, tokenizer)

# Debug: Check tokenized dataset format
# print("Tokenized train dataset sample:", train_dataset[0])
# print("Tokenized test dataset sample:", test_dataset[0])

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Train model with validation
model = train_model(train_dataset, test_dataset, tokenizer)

# Evaluate model
results = evaluate_model(model, test_dataset, tokenizer)

# Print results
print("\nEvaluation Results:")
print(f"Average BLEU Score: {results['bleu']:.4f}")
print(f"Average METEOR Score: {results['meteor']:.4f}")
print(f"Average BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"Average BERTScore Recall: {results['bertscore_recall']:.4f}")
print(f"Average BERTScore F1: {results['bertscore_f1']:.4f}")

# Print sample predictions
print("\nSample Predictions:")
for pred, ref in zip(results['predictions'][:3], results['references'][:3]):
    print(f"Predicted: {pred}")
    print(f"Reference: {ref}")
    print()

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Validation Epoch 1: 100%|██████████| 50/50 [00:05<00:00,  9.16it/s, batch_loss=0.454] 


New best model saved at epoch 1 with validation loss: 0.2610
Epoch 1: Training Loss = 3.9541, Validation Loss = 0.2610


Validation Epoch 2: 100%|██████████| 50/50 [00:05<00:00,  9.17it/s, batch_loss=0.397] 


...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.45 seconds, 111.53 sentences/sec

Evaluation Results:
Average BLEU Score: 0.5129
Average METEOR Score: 0.6912
Average BERTScore Precision: 0.9588
Average BERTScore Recall: 0.9619
Average BERTScore F1: 0.9603

Sample Predictions:
Predicted: As a user, I want to be able to select my preferred train seat layout so that I can easily navigate and book my preferred seat.
Reference: As a user, I want to be able to view the train seating layout, so that I can choose the seat that suits my preferences.

Predicted: As a customer, I want to be able to apply for loans online so that I don't have to go to a physical branch.
Reference: As a customer, I want to be able to apply for loans online so that I don’t have to go to a physical branch.

Predicted: As a user, I want to be able to leave feedback and reviews on properties so that other users can make informed decisions about whether to make a booking.
Reference: As a user, I want to be able to leave reviews of properties I have viewed s

In [7]:
# ==========================================================
#  NO FILTERING: generate on every input row, evaluate only where ref exists
# ==========================================================

import torch, gc
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score

# ------------------------------------------------
# CLEAR CUDA
# ------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("CUDA memory cleared.")
print("Device:", CONFIG['device'])

# ------------------------------------------------
# PATHS & HYPERPARAMS
# ------------------------------------------------
PROMISE_FILE = "/kaggle/input/promise-us/promise_us_full.csv"
PREDICTED_FILE = "/kaggle/input/predicted-fr/predicted_FR.csv"

OUTPUTS = {
    "GT→GEN":   "/kaggle/working/gt_gen_results.csv",
    "PRED→GEN": "/kaggle/working/pred_gen_results.csv",
    "NOISY→GEN":"/kaggle/working/noisy_pred_gen_results.csv",
    "RAW→GEN":  "/kaggle/working/raw_gen_all.csv",
}

BATCH_SIZE = 8
NOISE_RATE = 0.12   # 12% label noise

# ------------------------------------------------
# LOAD MODEL
# ------------------------------------------------
print("\nLoading T5 model...")
tokenizer = T5Tokenizer.from_pretrained(CONFIG['model_save_path'])
model = T5ForConditionalGeneration.from_pretrained(CONFIG['model_save_path'])
model.to(CONFIG['device'])
model.eval()
print("Model ready.")

# ------------------------------------------------
# GENERATORS: deterministic and noisy (sampling)
# ------------------------------------------------
def generate_batch(model, tokenizer, fr_texts):
    # deterministic beam search
    inputs = tokenizer(
        ["functional requirement: " + str(t) for t in fr_texts],
        max_length=CONFIG['max_input_length'],
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(CONFIG['device'])

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=CONFIG['max_target_length'] * 3,
            num_beams=6,
            min_length=20
        )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

def generate_batch_noisy(model, tokenizer, fr_texts):
    # stochastic sampling for robustness test
    inputs = tokenizer(
        ["functional requirement: " + str(t) for t in fr_texts],
        max_length=CONFIG['max_input_length'],
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(CONFIG['device'])

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=CONFIG['max_target_length'] * 3,
            do_sample=True,
            top_p=0.95,
            temperature=0.8,
            min_length=20
        )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

# ------------------------------------------------
# METRIC HELPERS
# ------------------------------------------------
smooth = SmoothingFunction().method1

def compute_ref_metrics(predictions, references):
    """
    Compute BLEU / METEOR / BERT-F1 on pairs where reference is present (non-empty).
    Also return counts: total_preds, n_evaluated_pairs.
    """
    total_preds = len(predictions)
    pairs = []
    for p, r in zip(predictions, references):
        p_str = "" if pd.isna(p) else str(p).strip()
        r_str = "" if pd.isna(r) else str(r).strip()
        if p_str and r_str:
            pairs.append((p_str, r_str))

    if not pairs:
        return {"BLEU":0.0,"METEOR":0.0,"BERT-F1":0.0, "n_total": total_preds, "n_eval": 0}

    preds, refs = zip(*pairs)

    bleu = np.mean([
        sentence_bleu([r.split()], p.split(),
                      weights=(0.25,0.25,0.25,0.25),
                      smoothing_function=smooth)
        for p,r in pairs
    ])

    meteor = np.mean([meteor_score([r.split()], p.split()) for p,r in pairs])

    P,R,F1 = score(list(preds), list(refs), lang="en", verbose=False)

    return {
        "BLEU": round(float(bleu),4),
        "METEOR": round(float(meteor),4),
        "BERT-F1": round(float(F1.mean().item()),4),
        "n_total": total_preds,
        "n_eval": len(pairs)
    }

# ------------------------------------------------
# NOISE INJECTION (ONCE)
# ------------------------------------------------
def add_balanced_noise(df, noise_rate=0.12):
    """
    Flip labels with higher probability for uncertain predictions.
    Keeps track of what was flipped for error-propagation analysis.
    """

    df = df.copy()

    # uncertainty is highest when pred_prob ≈ 0.5
    df["uncertainty"] = 1 - abs(df["pred_prob"] - 0.5) * 2  

    probs = df["uncertainty"] / df["uncertainty"].sum()
    n_flip = int(len(df) * noise_rate)

    np.random.seed(42)
    flip_idx = np.random.choice(
        df.index, size=n_flip, replace=False, p=probs
    )

    df["pred_label_noisy"] = df["pred_label"]
    df.loc[flip_idx, "pred_label_noisy"] = 1 - df.loc[flip_idx, "pred_label"]

    # keep traceability
    df["noise_flag"] = 0
    df.loc[flip_idx, "noise_flag"] = 1

    print(f"Injected noise into {n_flip} rows ({noise_rate*100:.1f}%)")

    return df

# ------------------------------------------------
# COLUMN DETECTION (REQUIRED HELPER)
# ------------------------------------------------
def detect_req_column(df):
    """
    Identify the functional requirement text column
    across different dataset formats.
    """
    if "RequirementText" in df.columns:
        return "RequirementText"
    elif "Functional Requirement" in df.columns:
        return "Functional Requirement"
    elif "FR" in df.columns:
        return "FR"
    else:
        raise ValueError(
            f"Could not detect requirement column. "
            f"Available columns: {df.columns.tolist()}"
        )

def compute_invest_all(predictions):
    # compute INVEST over all predictions (reference-free)
    return compute_invest(predictions)

def compute_qus_all(predictions):
    # compute QUS over all predictions (reference-free)
    return compute_qus(predictions)

# reuse user's invest & qus functions (they operate on predictions only)
def compute_invest(predictions):
    invest_scores = []
    for us in predictions:
        us = "" if pd.isna(us) else str(us).lower()
        score = 0
        if us.count(" and ") <= 1: score += 1
        if "shall" not in us and "must" not in us: score += 1
        if "so that" in us: score += 1
        if 10 <= len(us.split()) <= 35: score += 1
        action_verbs = ["view","create","edit","delete","access","save","display","submit","receive","track"]
        if any(v in us for v in action_verbs): score += 1
        invest_scores.append(score)
    return 0.0 if len(invest_scores)==0 else round(sum(invest_scores) / (5 * len(invest_scores)), 4)

def compute_qus(predictions):
    qus_scores = []
    for us in predictions:
        us = "" if pd.isna(us) else str(us).lower()
        score = 0
        if us.startswith("as a") or us.startswith("as an"): score += 1
        if "i want" in us: score += 1
        if "so that" in us: score += 1
        if 10 <= len(us.split()) <= 35: score += 1
        qus_scores.append(score)
    return 0.0 if len(qus_scores)==0 else round(sum(qus_scores) / (4 * len(qus_scores)), 4)

# ------------------------------------------------
# LOAD DATA
# ------------------------------------------------
promise_df = pd.read_csv(PROMISE_FILE, sep=";")
pred_df = pd.read_csv(PREDICTED_FILE).copy()
pred_df["pred_label_orig"] = pred_df["pred_label"]

# Inject noise ONCE
pred_df = add_balanced_noise(pred_df, NOISE_RATE)

comparison_results = []

# ------------------------------------------------
# 1) GT -> GEN  (generate on all promise rows, evaluate only where gold exists)
# ------------------------------------------------
print("\n🚀 RUNNING: GT→GEN (generate on all rows from PROMISE_FILE)")
gt_all = promise_df.copy()   # no filtering of input rows
req_col = detect_req_column(gt_all)

all_preds = []
for i in tqdm(range(0, len(gt_all), BATCH_SIZE)):
    batch = gt_all[req_col].iloc[i:i+BATCH_SIZE].tolist()
    all_preds.extend(generate_batch(model, tokenizer, batch))

gt_all["Generated User Story"] = all_preds
gt_all.to_csv(OUTPUTS["GT→GEN"], index=False)

# metrics: BLEU/METEOR/BERT on pairs with refs; INVEST/QUS over all preds
ref_metrics = compute_ref_metrics(gt_all["Generated User Story"].tolist(), gt_all["User Story"].tolist())
# --- REFERENCE-FREE METRICS ON ALL GENERATED STORIES ---
invest_all = compute_invest_all(gt_all["Generated User Story"].tolist())
qus_all = compute_qus_all(gt_all["Generated User Story"].tolist())


comparison_results.append({
    "Pipeline":"GT→GEN",
    "BLEU": ref_metrics["BLEU"],
    "METEOR": ref_metrics["METEOR"],
    "BERT-F1": ref_metrics["BERT-F1"],
    "INVEST": invest_all,
    "QUS": qus_all,
    "n_total": ref_metrics["n_total"],
    "n_eval": ref_metrics["n_eval"]
})

# ------------------------------------------------
# 2) PRED -> GEN  (generate on all predicted rows where pred_label==1 from PREDICTED_FILE)
# ------------------------------------------------
print("\n\n🚀 RUNNING: PRED→GEN (generate on all predicted FR rows from PREDICTED_FILE)")
df = pred_df.merge(
    promise_df[["RequirementText","User Story"]],
    on="RequirementText",
    how="left"
)
# Keep all rows from predicted set (we only restrict to pred_label==1 for this pipeline)
df_all = df[df["pred_label"]==1].reset_index(drop=True)

req_col = detect_req_column(df_all)
all_preds = []
for i in tqdm(range(0, len(df_all), BATCH_SIZE)):
    batch = df_all[req_col].iloc[i:i+BATCH_SIZE].tolist()
    all_preds.extend(generate_batch(model, tokenizer, batch))

df_all["Generated User Story"] = all_preds
df_all.to_csv(OUTPUTS["PRED→GEN"], index=False)

ref_metrics = compute_ref_metrics(df_all["Generated User Story"].tolist(), df_all["User Story"].tolist())
# --- REFERENCE-FREE METRICS ON ALL GENERATED STORIES ---
invest_all = compute_invest_all(gt_all["Generated User Story"].tolist())
qus_all = compute_qus_all(gt_all["Generated User Story"].tolist())

comparison_results.append({
    "Pipeline":"PRED→GEN",
    "BLEU": ref_metrics["BLEU"],
    "METEOR": ref_metrics["METEOR"],
    "BERT-F1": ref_metrics["BERT-F1"],
    "INVEST": invest_all,
    "QUS": qus_all,
    "n_total": ref_metrics["n_total"],
    "n_eval": ref_metrics["n_eval"]
})

# ------------------------------------------------
# 3) NOISY -> GEN  (use noisy labels; generate on those rows — stochastic decode)
# ------------------------------------------------
print("\n\n🚀 RUNNING: NOISY→GEN (generate on pred_label_noisy==1 with sampling)")
df_noisy = pred_df.merge(
    promise_df[["RequirementText","User Story"]],
    on="RequirementText",
    how="left"
)
df_noisy_all = df_noisy[df_noisy["pred_label_noisy"]==1].reset_index(drop=True)
req_col = detect_req_column(df_noisy_all)

all_preds = []
for i in tqdm(range(0, len(df_noisy_all), BATCH_SIZE)):
    batch = df_noisy_all[req_col].iloc[i:i+BATCH_SIZE].tolist()
    all_preds.extend(generate_batch_noisy(model, tokenizer, batch))

df_noisy_all["Generated User Story"] = all_preds
df_noisy_all.to_csv(OUTPUTS["NOISY→GEN"], index=False)

ref_metrics = compute_ref_metrics(df_noisy_all["Generated User Story"].tolist(), df_noisy_all["User Story"].tolist())
# --- REFERENCE-FREE METRICS ON ALL GENERATED STORIES ---
invest_all = compute_invest_all(gt_all["Generated User Story"].tolist())
qus_all = compute_qus_all(gt_all["Generated User Story"].tolist())


comparison_results.append({
    "Pipeline":"NOISY→GEN",
    "BLEU": ref_metrics["BLEU"],
    "METEOR": ref_metrics["METEOR"],
    "BERT-F1": ref_metrics["BERT-F1"],
    "INVEST": invest_all,
    "QUS": qus_all,
    "n_total": ref_metrics["n_total"],
    "n_eval": ref_metrics["n_eval"]
})

# robustness numbers
flipped_reached_gen = df_noisy_all["noise_flag"].sum() if "noise_flag" in df_noisy_all.columns else 0
lost_frs = len(pred_df[(pred_df["pred_label_orig"]==1) & (pred_df["pred_label_noisy"]==0)])
added_nfrs = len(pred_df[(pred_df["pred_label_orig"]==0) & (pred_df["pred_label_noisy"]==1)])

print("\n=== NOISY→GEN ROBUSTNESS SUMMARY ===")
print(f"Noise rate injected: {NOISE_RATE*100:.1f}%")
print(f"Rows sent to generator (noisy set total): {len(df_noisy_all)}")
print(f"Rows with refs evaluated (noisy set): {comparison_results[-1]['n_eval']}")
print(f"Lost true FRs due to noise: {lost_frs}")
print(f"Injected NFRs as FRs: {added_nfrs}")

# ------------------------------------------------
# 4) RAW -> GEN  (generate on all PROMISE rows; this is same as GT generation input-wise but we keep it as separate pipeline)
# ------------------------------------------------
print("\n\n🚀 RUNNING: RAW→GEN (generate on all rows from PROMISE_FILE - identical inputs to GT but kept for completeness)")
raw_all = promise_df.copy()
req_col = detect_req_column(raw_all)

all_preds = []
for i in tqdm(range(0, len(raw_all), BATCH_SIZE)):
    batch = raw_all[req_col].iloc[i:i+BATCH_SIZE].tolist()
    all_preds.extend(generate_batch(model, tokenizer, batch))

raw_all["Generated User Story"] = all_preds
raw_all.to_csv(OUTPUTS["RAW→GEN"], index=False)

ref_metrics = compute_ref_metrics(raw_all["Generated User Story"].tolist(), raw_all["User Story"].tolist())
invest_all = compute_invest_all(raw_all["Generated User Story"].tolist())
qus_all = compute_qus_all(raw_all["Generated User Story"].tolist())

comparison_results.append({
    "Pipeline":"RAW→GEN",
    "BLEU": ref_metrics["BLEU"],
    "METEOR": ref_metrics["METEOR"],
    "BERT-F1": ref_metrics["BERT-F1"],
    "INVEST": invest_all,
    "QUS": qus_all,
    "n_total": ref_metrics["n_total"],
    "n_eval": ref_metrics["n_eval"]
})

# ------------------------------------------------
# CONSOLIDATED DIAGNOSTICS
# ------------------------------------------------
print("\n\n=== CONSOLIDATED DIAGNOSTICS ===")

def read_meta(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

pred_gen  = read_meta(OUTPUTS["PRED→GEN"])
noisy_gen = read_meta(OUTPUTS["NOISY→GEN"])

# Basic counts and overlaps (note: sets only on RequirementText present in files)
gt_total = comparison_results[0]["n_total"]; gt_eval = comparison_results[0]["n_eval"]
pred_total = comparison_results[1]["n_total"]; pred_eval = comparison_results[1]["n_eval"]
noisy_total = comparison_results[2]["n_total"]; noisy_eval = comparison_results[2]["n_eval"]
raw_total = comparison_results[3]["n_total"]; raw_eval = comparison_results[3]["n_eval"]

print(f"GT→GEN   : total generated = {gt_total}, evaluated pairs = {gt_eval}")
print(f"PRED→GEN : total generated = {pred_total}, evaluated pairs = {pred_eval}")
print(f"NOISY→GEN: total generated = {noisy_total}, evaluated pairs = {noisy_eval}")
print(f"RAW→GEN  : total generated = {raw_total}, evaluated pairs = {raw_eval}")

# overlap checks for those with requirement text
def safe_set(df, col="RequirementText"):
    return set(df[col].dropna().astype(str).tolist()) if df is not None and col in df.columns else set()

gt_reqs  = safe_set(gt_all)
raw_reqs = safe_set(raw_all)
pred_reqs = safe_set(df_all)
noisy_reqs = safe_set(df_noisy_all)

print("\nGT ∩ RAW:", len(gt_reqs & raw_reqs))
print("PRED ∩ NOISY:", len(pred_reqs & noisy_reqs))

# if both saved CSVs exist, check identical generated stories for common RequirementText
if pred_gen is not None and noisy_gen is not None:
    merged = pred_gen.merge(noisy_gen, on="RequirementText", suffixes=("_pred","_noisy"))
    identical_pct = (merged["Generated User Story_pred"] == merged["Generated User Story_noisy"]).mean() if len(merged)>0 else float('nan')
    print(f"\n% identical generated stories (PRED vs NOISY) on overlap = {identical_pct:.3f}")
else:
    print("\n% identical generated stories (PRED vs NOISY): could not compute (missing files)")

# ------------------------------------------------
# FINAL TABLE (with totals & eval counts)
# ------------------------------------------------
results_df = pd.DataFrame(comparison_results)
# reorder columns for neatness
cols_order = ["Pipeline","BLEU","METEOR","BERT-F1","INVEST","QUS","n_total","n_eval"]
results_df = results_df[cols_order]
print("\n📊 FINAL PIPELINE TABLE (4 EXPERIMENTS)")
print(results_df.to_string(index=False))


CUDA memory cleared.
Device: cuda

Loading T5 model...
Model ready.
Injected noise into 30 rows (12.0%)

🚀 RUNNING: GT→GEN (generate on all rows from PROMISE_FILE)


100%|██████████| 79/79 [03:19<00:00,  2.52s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




🚀 RUNNING: PRED→GEN (generate on all predicted FR rows from PREDICTED_FILE)


100%|██████████| 33/33 [01:07<00:00,  2.06s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




🚀 RUNNING: NOISY→GEN (generate on pred_label_noisy==1 with sampling)


100%|██████████| 29/29 [00:36<00:00,  1.26s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== NOISY→GEN ROBUSTNESS SUMMARY ===
Noise rate injected: 12.0%
Rows sent to generator (noisy set total): 227
Rows with refs evaluated (noisy set): 227
Lost true FRs due to noise: 30
Injected NFRs as FRs: 0


🚀 RUNNING: RAW→GEN (generate on all rows from PROMISE_FILE - identical inputs to GT but kept for completeness)


100%|██████████| 79/79 [03:16<00:00,  2.49s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




=== CONSOLIDATED DIAGNOSTICS ===
GT→GEN   : total generated = 625, evaluated pairs = 256
PRED→GEN : total generated = 259, evaluated pairs = 259
NOISY→GEN: total generated = 227, evaluated pairs = 227
RAW→GEN  : total generated = 625, evaluated pairs = 256

GT ∩ RAW: 623
PRED ∩ NOISY: 224

% identical generated stories (PRED vs NOISY) on overlap = 0.050

📊 FINAL PIPELINE TABLE (4 EXPERIMENTS)
 Pipeline   BLEU  METEOR  BERT-F1  INVEST   QUS  n_total  n_eval
   GT→GEN 0.2482  0.4297   0.9282  0.7648 0.876      625     256
 PRED→GEN 0.2469  0.4281   0.9279  0.7648 0.876      259     259
NOISY→GEN 0.1959  0.3978   0.9181  0.7648 0.876      227     227
  RAW→GEN 0.2482  0.4297   0.9282  0.7648 0.876      625     256


In [17]:
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
from bert_score import score

def per_sample_metrics(df):
    scores = []
    preds = df["Generated User Story"].fillna("").astype(str).tolist()
    refs  = df["User Story"].fillna("").astype(str).tolist()

    # BERTScore once for all rows
    P, R, F1 = score(preds, refs, lang="en", verbose=False)

    for i, (p, r) in enumerate(zip(preds, refs)):
        if not p or not r:
            continue

        bleu = sentence_bleu([r.split()], p.split(),
                             weights=(0.25,0.25,0.25,0.25),
                             smoothing_function=lambda precisions, **kw: [pp+1e-12 for pp in precisions])

        meteor = meteor_score([r.split()], p.split())

        scores.append({
            "BLEU": bleu,
            "METEOR": meteor,
            "BERT-F1": float(F1[i].item())
        })
    return pd.DataFrame(scores)

# Load the saved outputs instead
gt_df   = pd.read_csv("/kaggle/working/gt_gen_results.csv")
pred_df = pd.read_csv("/kaggle/working/pred_gen_results.csv")
raw_df  = pd.read_csv("/kaggle/working/raw_gen_all.csv")
noisy_df = pd.read_csv("/kaggle/working/noisy_pred_gen_results.csv")

gt_scores   = per_sample_metrics(gt_df)
pred_scores = per_sample_metrics(pred_df)
raw_scores  = per_sample_metrics(raw_df)
noisy_scores = per_sample_metrics(noisy_df)


def bootstrap_mean_std(df, n=2000):
    means = []
    for _ in range(n):
        sample = df.sample(frac=1, replace=True)
        means.append(sample.mean())
    return pd.DataFrame(means).mean(), pd.DataFrame(means).std()

gt_mean, gt_std   = bootstrap_mean_std(gt_scores)
pred_mean, pred_std = bootstrap_mean_std(pred_scores)
raw_mean, raw_std = bootstrap_mean_std(raw_scores)

summary = pd.DataFrame({
    "Pipeline": ["GT→GEN","PRED→GEN","RAW→GEN"],
    "BLEU_mean":[gt_mean.BLEU, pred_mean.BLEU, raw_mean.BLEU],
    "BLEU_std":[gt_std.BLEU, pred_std.BLEU, raw_std.BLEU],
    "METEOR_mean":[gt_mean.METEOR, pred_mean.METEOR, raw_mean.METEOR],
    "METEOR_std":[gt_std.METEOR, pred_std.METEOR, raw_std.METEOR],
    "BERTF1_mean":[gt_mean["BERT-F1"], pred_mean["BERT-F1"], raw_mean["BERT-F1"]],
    "BERTF1_std":[gt_std["BERT-F1"], pred_std["BERT-F1"], raw_std["BERT-F1"]],
})

print("\n=== Mean ± Std (bootstrapped) ===")
print(summary.round(4))


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho


=== Mean ± Std (bootstrapped) ===
   Pipeline  BLEU_mean  BLEU_std  METEOR_mean  METEOR_std  BERTF1_mean  BERTF1_std
0    GT→GEN     0.2466    0.0087       0.4295      0.0078       0.9282      0.0011
1  PRED→GEN     0.2458    0.0090       0.4282      0.0081       0.9279      0.0011
2   RAW→GEN     0.2467    0.0089       0.4297      0.0080       0.9282      0.0011


In [18]:
# ==========================================================
# === ERROR PROPAGATION INSIGHTS (correctly aligned) ===
# ==========================================================

# Confusion counts based on original vs noisy labels
tp = len(pred_df[(pred_df["pred_label_orig"] == 1) &
                 (pred_df["pred_label_noisy"] == 1)])

fp = len(pred_df[(pred_df["pred_label_orig"] == 0) &
                 (pred_df["pred_label_noisy"] == 1)])

fn = len(pred_df[(pred_df["pred_label_orig"] == 1) &
                 (pred_df["pred_label_noisy"] == 0)])

print("\n=== Error Propagation Insights ===")
print(f"{fp} false positives were incorrectly sent to the generator.")
print(f"{fn} true FRs were lost and never converted to user stories.")

fp_ratio = fp / (tp + fp) if (tp + fp) > 0 else 0
fn_ratio = fn / (tp + fn) if (tp + fn) > 0 else 0

print(f"False-positive rate: {round(fp_ratio,3)}")
print(f"False-negative rate: {round(fn_ratio,3)}")



=== Error Propagation Insights ===
0 false positives were incorrectly sent to the generator.
32 true FRs were lost and never converted to user stories.
False-positive rate: 0.0
False-negative rate: 0.124


In [19]:
# ==========================================================
# === HUMAN CHECK SAMPLE TABLES (correct variables) ===
# ==========================================================

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_rows", 20)

def make_sample_table(name, df):
    req_col = detect_req_column(df)

    sample = df.sample(min(10, len(df)), random_state=42).reset_index(drop=True)

    table = pd.DataFrame({
        "Pipeline": name,
        "Requirement": sample[req_col],
        "Generated_User_Story": sample["Generated User Story"]
    })

    print(f"\n========== {name} — HUMAN CHECK SAMPLE (10 rows) ==========\n")
    display(table)
    return table

# Generate tables for all three pipelines
gt_sample   = make_sample_table("GT→GEN",   gt_all)
pred_sample = make_sample_table("PRED→GEN", df_all)
raw_sample  = make_sample_table("RAW→GEN",  raw_all)

# Combine for appendix
all_samples = pd.concat([gt_sample, pred_sample, raw_sample], ignore_index=True)

print("\n========== COMBINED HUMAN CHECK TABLE ==========\n")
display(all_samples)

# Save for appendix / supplementary material
all_samples.to_csv("/kaggle/working/human_check_samples.csv", index=False)



========== GT→GEN — HUMAN CHECK SAMPLE (10 rows) ==========



,Pipeline,Requirement,Generated_User_Story
0,GT→GEN,viewing a movie details the website will display the movies description actor and director entered in by the administrator.,"As a user, I want to be able to view movie details such as actor and director so that I can make an informed decision."
1,GT→GEN,The leads washing functionality will have an interface in which lead data parameters can be maintained.,"As a lead washer, I want to be able to manage lead data, so that I can ensure that leads are being washed as often as possible."
2,GT→GEN,The user shall easily locate instructions while using the product. User help can be found within 90% of the system.,"As a user, I want to be able to easily locate instructions while using the product, so that I can easily follow the instructions."
3,GT→GEN,The product shall increase productivity of Collision Estimators. 80% of the Collision Estimators shall agree their productivity has increase within 1 month of using the product.,"As a Collision Estimator, I want to be able to increase productivity of Collision Estimators."
4,GT→GEN,If the leads score falls within the medium average then it will be set for manual verification by an Enrollment Coordinator through the eleads system.,"As a Lead Coordinator, I want to be able to automatically verify if a lead's score falls within the medium average so that I can ensure they are eligible to enroll."
5,GT→GEN,Once a game is initiated the product shall allow each player to position their 5 ships on their respective defensive grids.,"As a player, I want to be able to position my 5 ships on my respective defensive grids."
6,GT→GEN,Only users with Update access must be able to initiate dispute requests.,"As a user, I want to be able to initiate dispute requests only from users with Update access."
7,GT→GEN,The response time of general student management tasks shall take no longer than 5 seconds and the response time of schedule generation shall take no longer than 30 seconds.,"As a student, I want the response time of general student management tasks to take no longer than 5 seconds and the response time of schedule generation to take no longer than 30 seconds."
8,GT→GEN,For each shot the product shall mark the shot as a hit or miss on the defensive grid of the defensive player.,"As a player, I want to be able to mark shots as hit or miss on the defensive grid of the defensive player."
9,GT→GEN,Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Clinical Site including the clinical site name site contact person contact information and address.,"As an Administrator, I want to be able to modify information relating to a Clinical Site including the clinical site name site contact person contact information and address."



========== PRED→GEN — HUMAN CHECK SAMPLE (10 rows) ==========



,Pipeline,Requirement,Generated_User_Story
0,PRED→GEN,The system shall be able to display a printable summary for individual nursing students which will include (but not be limited to) student name student ID admission date classes credits GPA and the cohort that the student is enrolled in.,"As a student, I want to be able to print a summary for individual nursing students so that I can keep track of my progress."
1,PRED→GEN,A non-clinical class shall specify the course name lecture room requirements and instructor needs.,"As a non-clinical student, I want to be able to specify course room requirements and instructor needs so that I can plan my schedule accordingly."
2,PRED→GEN,The system shall color code events according to their variance from current time.,"As a user, I want to be able to color code events according to their variance from current time."
3,PRED→GEN,The system will notify affected parties when changes occur affecting cohorts including but not limited to changes to the sequence for a cohort’s program of study and changes to a given week’s schedule (lab cancelled this week due to instructor illness).,"As a student, I want to be able to receive notifications when changes occur in my cohort’s program of study so that I can stay informed."
4,PRED→GEN,The ratings shall be from a scale of 1-10,"As a user, I want to be able to rate the usability of this site as a whole, so that I can make an informed decision about whether to upgrade or not."
5,PRED→GEN,Any disputes cases that have been closed for over 6 months must be purged from the online disputes database.,"As a user, I want to be able to delete any disputes cases that have been closed for over 6 months."
6,PRED→GEN,Vendor will submit a credit validation record to be applied to the enterprise billing system.,"As a vendor, I want to be able to submit a credit validation record to the enterprise billing system, so that we can validate the vendor's credit history."
7,PRED→GEN,Once a game is initiated the product shall display an offensive grid to each player.,"As a gamer, I want to be able to view offensive grids and defensive grids so that I can plan my strategy."
8,PRED→GEN,The recycled parts search results provided to the estimator shall be retrieved by the system.,"As a user, I want to be able to retrieve the recycled parts search results provided by the estimator."
9,PRED→GEN,If the leads score falls within the low average then it will be returned to the supplying vendor,"As a lead owner, I want to be able to view my leads score, so that I can ensure that I am getting quality leads."



========== RAW→GEN — HUMAN CHECK SAMPLE (10 rows) ==========



,Pipeline,Requirement,Generated_User_Story
0,RAW→GEN,viewing a movie details the website will display the movies description actor and director entered in by the administrator.,"As a user, I want to be able to view movie details such as actor and director so that I can make an informed decision."
1,RAW→GEN,The leads washing functionality will have an interface in which lead data parameters can be maintained.,"As a lead washer, I want to be able to manage lead data, so that I can ensure that leads are being washed as often as possible."
2,RAW→GEN,The user shall easily locate instructions while using the product. User help can be found within 90% of the system.,"As a user, I want to be able to easily locate instructions while using the product, so that I can easily follow the instructions."
3,RAW→GEN,The product shall increase productivity of Collision Estimators. 80% of the Collision Estimators shall agree their productivity has increase within 1 month of using the product.,"As a Collision Estimator, I want to be able to increase productivity of Collision Estimators."
4,RAW→GEN,If the leads score falls within the medium average then it will be set for manual verification by an Enrollment Coordinator through the eleads system.,"As a Lead Coordinator, I want to be able to automatically verify if a lead's score falls within the medium average so that I can ensure they are eligible to enroll."
5,RAW→GEN,Once a game is initiated the product shall allow each player to position their 5 ships on their respective defensive grids.,"As a player, I want to be able to position my 5 ships on my respective defensive grids."
6,RAW→GEN,Only users with Update access must be able to initiate dispute requests.,"As a user, I want to be able to initiate dispute requests only from users with Update access."
7,RAW→GEN,The response time of general student management tasks shall take no longer than 5 seconds and the response time of schedule generation shall take no longer than 30 seconds.,"As a student, I want the response time of general student management tasks to take no longer than 5 seconds and the response time of schedule generation to take no longer than 30 seconds."
8,RAW→GEN,For each shot the product shall mark the shot as a hit or miss on the defensive grid of the defensive player.,"As a player, I want to be able to mark shots as hit or miss on the defensive grid of the defensive player."
9,RAW→GEN,Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Clinical Site including the clinical site name site contact person contact information and address.,"As an Administrator, I want to be able to modify information relating to a Clinical Site including the clinical site name site contact person contact information and address."



========== COMBINED HUMAN CHECK TABLE ==========



,Pipeline,Requirement,Generated_User_Story
0,GT→GEN,viewing a movie details the website will display the movies description actor and director entered in by the administrator.,"As a user, I want to be able to view movie details such as actor and director so that I can make an informed decision."
1,GT→GEN,The leads washing functionality will have an interface in which lead data parameters can be maintained.,"As a lead washer, I want to be able to manage lead data, so that I can ensure that leads are being washed as often as possible."
2,GT→GEN,The user shall easily locate instructions while using the product. User help can be found within 90% of the system.,"As a user, I want to be able to easily locate instructions while using the product, so that I can easily follow the instructions."
3,GT→GEN,The product shall increase productivity of Collision Estimators. 80% of the Collision Estimators shall agree their productivity has increase within 1 month of using the product.,"As a Collision Estimator, I want to be able to increase productivity of Collision Estimators."
4,GT→GEN,If the leads score falls within the medium average then it will be set for manual verification by an Enrollment Coordinator through the eleads system.,"As a Lead Coordinator, I want to be able to automatically verify if a lead's score falls within the medium average so that I can ensure they are eligible to enroll."
...,...,...,...
25,RAW→GEN,Once a game is initiated the product shall allow each player to position their 5 ships on their respective defensive grids.,"As a player, I want to be able to position my 5 ships on my respective defensive grids."
26,RAW→GEN,Only users with Update access must be able to initiate dispute requests.,"As a user, I want to be able to initiate dispute requests only from users with Update access."
27,RAW→GEN,The response time of general student management tasks shall take no longer than 5 seconds and the response time of schedule generation shall take no longer than 30 seconds.,"As a student, I want the response time of general student management tasks to take no longer than 5 seconds and the response time of schedule generation to take no longer than 30 seconds."
28,RAW→GEN,For each shot the product shall mark the shot as a hit or miss on the defensive grid of the defensive player.,"As a player, I want to be able to mark shots as hit or miss on the defensive grid of the defensive player."


In [21]:
from scipy.stats import ttest_rel   # <-- REQUIRED IMPORT

# Align same indices for fair comparison
common_idx = gt_scores.index.intersection(pred_scores.index)

stat, pval = ttest_rel(
    gt_scores.loc[common_idx, "BERT-F1"],
    pred_scores.loc[common_idx, "BERT-F1"]
)

print("\n=== Paired t-test (GT vs PRED) ===")
print(f"t-statistic = {stat:.4f}")
print(f"p-value     = {pval:.6f}")



=== Paired t-test (GT vs PRED) ===
t-statistic = 0.1420
p-value     = 0.887175


In [22]:
final_table = summary.copy()
print("\n=== FINAL REVIEWER-READY TABLE ===")
print(final_table.round(4).to_string(index=False))



=== FINAL REVIEWER-READY TABLE ===
Pipeline  BLEU_mean  BLEU_std  METEOR_mean  METEOR_std  BERTF1_mean  BERTF1_std
  GT→GEN     0.2466    0.0087       0.4295      0.0078       0.9282      0.0011
PRED→GEN     0.2458    0.0090       0.4282      0.0081       0.9279      0.0011
 RAW→GEN     0.2467    0.0089       0.4297      0.0080       0.9282      0.0011
